# Score transformations and pseudo-labels

Apply the public transformation helpers to a small deterministic dataset with an observed wild-type row. Each helper returns a new DataFrame.

In [ ]:
import pandas as pd

from dmsroute import (
    add_minmax,
    add_pseudo_binary_label,
    add_wt_relative_score,
    add_zscore,
    compute_wt_score,
)

In [ ]:
scores = pd.DataFrame(
    {
        "variant": ["WT", "A1V", "A1G", "A1L", "A1P"],
        "is_wildtype": [True, False, False, False, False],
        "score_raw": [1.0, 1.5, 1.1, 0.8, 0.4],
    }
)
scores

## WT reference and relative methods

`ratio` requires a non-zero WT score. `log_ratio` and `log2_ratio` additionally require every score-to-WT ratio to be strictly positive. `difference` accepts any finite scores.

In [ ]:
wt_score = compute_wt_score(scores, score_col="score_raw")
wt_score

In [ ]:
relative = add_wt_relative_score(
    scores, "score_raw", method="ratio", output_col="score_ratio"
)
relative = add_wt_relative_score(
    relative, "score_raw", method="log_ratio", output_col="score_log_ratio"
)
relative = add_wt_relative_score(
    relative, "score_raw", method="log2_ratio", output_col="score_log2_ratio"
)
relative = add_wt_relative_score(
    relative, "score_raw", method="difference", output_col="score_difference"
)
relative

## Pseudo-binary labels

With `higher_is_better=True`, values above `+delta` receive 1, values below `-delta` receive 0, and the inclusive neutral interval receives the chosen label (999 here). Set `higher_is_better=False` to reverse the two non-neutral classes.

In [ ]:
labeled = add_pseudo_binary_label(
    relative,
    score_col="score_difference",
    delta=0.2,
    neutral_label=999,
    higher_is_better=True,
    output_col="higher_is_better_label",
)
labeled = add_pseudo_binary_label(
    labeled,
    score_col="score_difference",
    delta=0.2,
    neutral_label=999,
    higher_is_better=False,
    output_col="lower_is_better_label",
)
labeled.loc[
    :,
    [
        "variant",
        "score_difference",
        "higher_is_better_label",
        "lower_is_better_label",
    ],
]

## Dataset-wide scaling

Z-score standardization uses the sample standard deviation; min-max scaling maps the observed range to `[0, 1]`.

In [ ]:
scaled = add_zscore(
    labeled, "score_raw", output_col="score_zscore"
)
scaled = add_minmax(
    scaled, "score_raw", output_col="score_minmax"
)
scaled.loc[:, ["variant", "score_raw", "score_zscore", "score_minmax"]]